# Asteroids RAM Environment and a Hard-Coded Policy

> **Overview.** Atari Asteroids with 128-byte RAM observations in any game mode and difficulty, and a hand-coded policy
> to play it, as a data source for learning world models.

**Env.** ALE Asteroids, minimal action set (14 actions). Each step repeats the action for 4 frames. There are no sticky
actions, so the game is deterministic given the actions.
- **Modes** 0–31 are the one-player games. Bit 0 makes small and medium asteroids faster. Bits 1–2 set the bonus ship
  (every 5k / 10k / 20k points / never). Bits 3–4 set what DOWN does (hyperspace / shields / flip / nothing). Mode 128
  is the children's game: 4 asteroids per wave, and large asteroids split into one medium.
- **Difficulty** 0 is the normal game; 3 adds UFOs and satellites that shoot at the ship.

**Policy.** Each step, pick a target asteroid with probability ∝ exp(−distance / 10 px) (across the screen wraps),
turn towards it, and fire every other step, since holding FIRE only shoots once. When no asteroid is within 60 px,
also thrust with probability 0.1, so the ship roams the screen. It never presses DOWN.

In [1]:
import cv2
import imageio
import numpy as np
from ale_py import ALEInterface, LoggerMode, roms
from IPython.display import Video

ALEInterface.setLoggerMode(LoggerMode.Error)
FRAMESKIP = 4


class AsteroidsRAM:
    def __init__(self, mode=0, difficulty=0, seed=0):
        self.ale = ALEInterface()
        self.ale.setInt('random_seed', seed)
        self.ale.setFloat('repeat_action_probability', 0.0)
        self.ale.loadROM(roms.Asteroids)
        self.ale.setMode(mode)
        self.ale.setDifficulty(difficulty)
        self.actions = self.ale.getMinimalActionSet()

    def reset(self):
        self.ale.reset_game()
        self.prev_screen = self.ale.getScreenRGB()
        return self.ale.getRAM().copy()

    def step(self, a):
        reward = 0
        for k in range(FRAMESKIP):
            if k == FRAMESKIP - 1:
                self.prev_screen = self.ale.getScreenRGB()
            reward += self.ale.act(self.actions[a])
        return self.ale.getRAM().copy(), reward, self.ale.game_over()

    def render(self):
        # Asteroids and the ship are drawn on alternate frames; the max over the last two shows both.
        return np.maximum(self.prev_screen, self.ale.getScreenRGB())

In [2]:
# RAM layout (checked against the screen). Asteroids are in two lists, y at 3.. and 12.., each ending at y == 224, with
# x at +18 and size/colour at +36. Ship x, y at 73, 74 (y == 224 while hidden); heading in the low nibble of 60.
SIZES = [(16, 28), (8, 14), (4, 8)]  # large, medium, small (w, h) in pixels
WRAP = np.array([160, 178])  # the screen wraps every 160 px across and 178 px down


def x_pixel(v):
    # Atari coarse/fine horizontal position byte -> pixel column (decoding from OC_Atari).
    if v <= 1:
        return 64 if v == 0 else 4
    lo, add, sub = v & 15, 8 * (v >> 7), (v >> 4) & 7
    base = 97 + 15 * (lo // 2 - 1) if lo % 2 == 0 else 10 + 15 * ((lo - 1) // 2 - 1)
    return base + add - sub


def signed(v):
    return v - 256 if v > 127 else v


def decode(ram):
    # Ship centre (None while hidden), heading 0-15 (0 = up, counter-clockwise), asteroid centres (n, 2), in pixels.
    r = ram.astype(int)
    asteroids = []
    for start in (3, 12):
        for i in range(start, start + 8):
            if r[i] == 224:
                break
            c = r[i + 36] & 127
            w, h = SIZES[0 if c < 32 else 1 if c < 48 else 2]
            asteroids.append((x_pixel(r[i + 18]) + w / 2, 24 + 2 * signed(r[i]) + h / 2))
    ship = None if r[74] == 224 else np.array([x_pixel(r[73]) + 2.5, 23 + 2 * signed(r[74])])
    return ship, r[60] & 15, np.array(asteroids).reshape(-1, 2)

In [3]:
ACTIONS = ['NOOP', 'FIRE', 'UP', 'RIGHT', 'LEFT', 'DOWN', 'UPRIGHT', 'UPLEFT', 'UPFIRE', 'RIGHTFIRE', 'LEFTFIRE',
           'DOWNFIRE', 'UPRIGHTFIRE', 'UPLEFTFIRE']  # the minimal action set, in order
TEMPERATURE = 10  # px; the target is sampled with probability ∝ exp(-distance / TEMPERATURE)
SAFE = 60  # px; thrust only when no asteroid is closer than this
THRUST_PROB = 0.1


def policy(ram, t, rng):
    ship, heading, asteroids = decode(ram)
    if ship is None or len(asteroids) == 0:
        return ACTIONS.index('NOOP')
    d = (asteroids - ship + WRAP / 2) % WRAP - WRAP / 2
    dist = np.hypot(*d.T)
    p = np.exp(-(dist - dist.min()) / TEMPERATURE)
    dx, dy = d[rng.choice(len(d), p=p / p.sum())]
    # The 16 headings are evenly spaced when y is measured in RAM units (2 px).
    turn = (round(np.arctan2(-dx, -dy / 2) / (np.pi / 8)) - heading) % 16
    thrust = dist.min() > SAFE and rng.random() < THRUST_PROB
    name = ('UP' if thrust else '') + ('' if turn == 0 else 'LEFT' if turn <= 8 else 'RIGHT') + ('FIRE' if t % 2 == 0 else '')
    return ACTIONS.index(name or 'NOOP')

In [4]:
CONFIGS = [(0, 0), (1, 0), (128, 0), (0, 3), (1, 3), (128, 3)]  # (mode, difficulty)
MODE_NAMES = {0: 'slow', 1: 'fast', 128: 'kids'}
STEPS = 2000


def panel(screen, mode, difficulty, done):
    p = np.zeros((448, 320, 3), np.uint8)
    p[28:] = cv2.resize(screen, (320, 420), interpolation=cv2.INTER_NEAREST)
    label = f'mode {mode} ({MODE_NAMES[mode]})  diff {difficulty}' + ('  GAME OVER' if done else '')
    cv2.putText(p, label, (6, 19), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 1, cv2.LINE_AA)
    cv2.rectangle(p, (0, 0), (319, 447), (90, 90, 90), 1)
    return p


envs = [AsteroidsRAM(m, d) for m, d in CONFIGS]
rams = [env.reset() for env in envs]
rngs = [np.random.default_rng(i) for i in range(len(envs))]
scores, done = [0] * len(envs), [False] * len(envs)
with imageio.get_writer('policy.mp4', fps=30) as video:  # 2x real time
    for t in range(STEPS):
        panels = []
        for i, env in enumerate(envs):
            if not done[i]:
                rams[i], r, done[i] = env.step(policy(rams[i], t, rngs[i]))
                scores[i] += r
            panels.append(panel(env.render(), *CONFIGS[i], done[i]))
        video.append_data(np.vstack([np.hstack(panels[:3]), np.hstack(panels[3:])]))
print('scores after', STEPS, 'steps:', dict(zip(CONFIGS, scores)))
Video('policy.mp4', embed=True, html_attributes='controls loop autoplay muted')

scores after 2000 steps: {(0, 0): 9730, (1, 0): 9940, (128, 0): 6020, (0, 3): 6730, (1, 3): 9360, (128, 3): 4740}
